In [ ]:
import requests
from datetime import datetime
from bs4 import BeautifulSoup
import re, random, time, pandas as pd, os, json
from rake_nltk import Rake
import nltk
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt')

#Configuration Block----------
Output_Filename = r"D:\My_Projects\Project_Business_Analyst\Raw_Data\Fashion_Men.csv"
Target_URL = "https://www.amazon.in/s?i=apparel&rh=n%3A1968120031&s=popularity-rank&fs=true&page="
Category = "Fashion"
SubCategory = "Mens Shirts"
Start_Page_Num = 1
End_Page_Num = 2
# ------------------------------


MAX_RETRIES = 5
DELAY_RANGE = (2, 8) 
BASE_URL = Target_URL
OUTPUT_FILE = Output_Filename

def get_headers():
    user_agents = [
        # Current Chrome on Windows (2024-2025)
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/126.0.0.0 Safari/537.36",
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36",
        # Current Firefox
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:127.0) Gecko/20100101 Firefox/127.0",
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:126.0) Gecko/20100101 Firefox/126.0",
        # Current Chrome on Mac
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/126.0.0.0 Safari/537.36",
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36",
        # Edge
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/126.0.0.0 Safari/537.36 Edg/126.0.0.0",
    ]
    return {
        "User-Agent": random.choice(user_agents),
        "Accept-Language": "en-GB,en;q=0.9",
        "Accept-Encoding": "gzip, deflate, br",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
        "DNT": "1",
        "Connection": "keep-alive",
        "Upgrade-Insecure-Requests": "1",
        "Referer": "https://www.amazon.in/",
        "Sec-Fetch-Site": "same-origin",
        "Sec-Fetch-Mode": "navigate",
        "Sec-Fetch-User": "?1",
        "Sec-Fetch-Dest": "document",
        "Upgrade-Insecure-Requests": "1"
    }

def make_request(url):
    for attempt in range(MAX_RETRIES):
        try:
            session = requests.Session()
            headers = get_headers()
            session.headers.update(headers)
            wait_time = random.uniform(*DELAY_RANGE) + attempt * 4
            time.sleep(wait_time)
            response = session.get(url, timeout=15)
            if response.status_code == 200:
                html = response.text
                if any(bot_text in html for bot_text in [
                    "api-services-support@amazon.com",
                    "Robot Check",
                    "Type the characters you see in this image",
                    "<title></title>"
                ]):
                    print("[WARNING] Bot detection triggered.")
                    continue
                return response
            else:
                print(f"[WARNING] Status code: {response.status_code}")
        except Exception as e:
            print(f"[Retry {attempt+1}] Error: {e}")
    print(f"[ERROR] No response from: {url}")
    return None

def get_product_links(soup):
    links = set()
    for a in soup.select('a.a-link-normal.s-no-outline,a.a-link-normal.a-text-normal'):
        href = a.get('href', '')
        if '/dp/' in href or '/gp/product/' in href:
            full_url = f"https://www.amazon.in{href}" if href.startswith('/') else href
            clean_url = re.sub(r'/ref=.*', '', full_url).split('?')[0]
            links.add(clean_url)
    return list(links)

def get_title(soup):
    title = soup.select_one("span#productTitle, h1.a-size-large, span.a-size-large.product-title-word-break")
    return title.get_text(strip=True) if title else " "

def get_price_from_search(soup):
    try:
        price_block = soup.find("span", class_="a-price")
        if price_block:
            price_whole = price_block.find("span", class_="a-price-whole")
            if price_whole:
                whole = price_whole.text.strip().replace(",", "")
                return whole

        offscreen_prices = soup.find_all("span", class_="a-offscreen")
        prices = []

        for span in offscreen_prices:
            text = span.text.strip().replace("₹", "").replace(",", "")
            if text.replace(".", "").isdigit():  
                prices.append(float(text))

        if prices:
            lowest_price = int(min(prices))
            return str(lowest_price)

        return " "
    except Exception as e:
        print(f"Error extracting price: {e}")
        return " "
    
def get_amazon_images(url):
    for attempt in range(MAX_RETRIES):
        try:
            session = requests.Session()
            headers = get_headers()
            session.headers.update(headers)

            wait_time = random.uniform(*DELAY_RANGE) + attempt * 2
            time.sleep(wait_time)

            response = session.get(url, timeout=10)
            if response.status_code != 200:
                continue

            soup = BeautifulSoup(response.content, 'html.parser')
            image_urls = set()

            dynamic_image = soup.find('img', class_='a-dynamic-image')
            if dynamic_image:
                data = dynamic_image.get('data-a-dynamic-image')
                if data:
                    image_urls.update(list(json.loads(data).keys()))

            thumbnails = soup.select('#altImages img')
            for img in thumbnails:
                src = img.get('src')
                if src and "data:image" not in src:
                    high_res = re.sub(r'\._[A-Z0-9_,]+_\.', '._SL1500_.', src)
                    image_urls.add(high_res)

            scripts = soup.find_all("script")
            for script in scripts:
                if script.string and "ImageBlockATF" in script.string:
                    matches = re.findall(r'https://[\w./-]+\.jpg', script.string)
                    image_urls.update(matches)

            if image_urls:
                return list(image_urls)[:8]

        except Exception as e:
            print(f"[Attempt {attempt + 1}] Image scraping error: {e}")

    return []

def extract_about_this_item(soup):
    about_section = soup.find('h3', string=lambda text: text and "About this item" in text)
    if about_section:
        ul = about_section.find_next('ul')
        if ul:
            bullets = ul.find_all('li')
            descriptions = [li.get_text(strip=True) for li in bullets if li.get_text(strip=True)]
            if descriptions:
                return ' | '.join(descriptions)

    feature_div = soup.find('div', id='feature-bullets')
    if feature_div:
        bullets = feature_div.find_all('span', class_='a-list-item')
        descriptions = [b.get_text(strip=True) for b in bullets if b.get_text(strip=True)]
        if descriptions:
            return ' | '.join(descriptions)

    uls = soup.find_all('ul')
    longest_ul = max(uls, key=lambda ul: len(ul.find_all('li')), default=None)
    if longest_ul:
        descriptions = [li.get_text(strip=True) for li in longest_ul.find_all('li') if li.get_text(strip=True)]
        if descriptions:
            return ' | '.join(descriptions)

    return ''

def remove_non_english_words(text):
    words = text.split()
    english_words = [w for w in words if re.match("^[A-Za-z0-9\-]+$", w)]
    return ' '.join(english_words)

def get_keywords(description, max_words=20):
    rake = Rake()
    rake.extract_keywords_from_text(description)
    keywords = rake.get_ranked_phrases()[:max_words]
    return ', '.join(keywords)

def get_product_details(url, sr_no, Category, SubCategory):
    response = make_request(url)
    if not response:
        return None

    soup = BeautifulSoup(response.content, 'html.parser')
    product = {
        
        "Sr.No": sr_no,
        "Scraped_Date": " ",
        "uid": re.search(r'/dp/([A-Z0-9]{10})', url).group(1) if re.search(r'/dp/([A-Z0-9]{10})', url) else '',
        "url": url,
        "title": " ",
        "description": " ",
        "category": Category,
        "subcategory": SubCategory,
        "rating": " ", 
        "review_count": " ",
        "price": get_price_from_search(soup),   
    }

    product["title"] = remove_non_english_words(get_title(soup))
    if not product["title"]:
        print(f"[WARNING] Empty title for URL: {url}")
    if not soup.select_one('#productTitle'):
        print("[DEBUG] productTitle not found in soup")
        print(soup.prettify()[:1000])  

    raw_description = extract_about_this_item(soup)
    product["description"] = remove_non_english_words(raw_description)

    # ========================================================
    # SEPARATE SECTION: BRAND EXTRACTION (UPDATED)
    # ========================================================
    # 1. Initialize the field in your dictionary
    product['brand'] = " "
    
    # --- METHOD 1: YOUR ORIGINAL LOGIC ---
    # Look for the <th> tag that explicitly contains "Brand" or "Brand Name"
    brand_th = soup.find('th', string=lambda text: text and "brand" in text.lower())
    
    if brand_th:
        # Find the adjacent <td> element holding the brand value
        brand_td = brand_th.find_next_sibling('td')
        if brand_td:
            raw_brand = brand_td.get_text(strip=True)
            # Clean up any hidden unicode formatting marks
            clean_brand = raw_brand.replace('\u200e', '').replace('\u200f', '').strip()
            product['brand'] = clean_brand
    
    # --- METHOD 2: FALLBACK (Runs only if Method 1 found nothing or "Not found") ---
    if product['brand'] == " " or product['brand'] == "Not found":
        # Loop through all <th> elements dynamically to catch variations like "Brand Name "
        for th in soup.find_all('th'):
            th_text = th.get_text(strip=True).lower()
            
            # Check if the text matches either "brand" or "brand name" flexibly
            if "brand" in th_text:
                brand_td = th.find_next_sibling('td')
                if brand_td:
                    raw_brand = brand_td.get_text(strip=True)
                    clean_brand = raw_brand.replace('\u200e', '').replace('\u200f', '').strip()
                    product['brand'] = clean_brand
                    break # Stop once found
    
    # Final safety check if absolutely nothing worked
    if product['brand'] == " ":
        product['brand'] = "Not found"


    # ========================================================
    # SEPARATE SECTION: MANUFACTURER EXTRACTION
    # ========================================================
    # 1. Initialize the field in your dictionary
    product['Manufacturer'] = " "
    
    # 2. Find the technical details table row containing the Manufacturer info
    # We search for the <th> tag that explicitly contains the text "Manufacturer"
    manufacturer_th = soup.find('th', string=lambda text: text and "manufacturer" in text.lower())
    
    if manufacturer_th:
        # 3. Find the next sibling <td> element which holds the value (e.g., "HP")
        manufacturer_td = manufacturer_th.find_next_sibling('td')
        if manufacturer_td:
            raw_manufacturer = manufacturer_td.get_text(strip=True)
            
            # Clean up any weird invisible spaces or escape characters (like &lrm; seen in your DOM)
            clean_manufacturer = raw_manufacturer.replace('\u200e', '').replace('\u200f', '').strip()
            product['Manufacturer'] = clean_manufacturer
    else:
        # Fallback to an alternate selector if the table style changes on other pages
        manufacturer_span = soup.select_one('span.a-size-base.prodDetAttrValue') # Be careful, this can be generic
        if manufacturer_span:
            product['Manufacturer'] = manufacturer_span.get_text(strip=True)
        else:
            product['Manufacturer'] = "Not found"


# ========================================================
    # SEPARATE SECTION: M.R.P. EXTRACTION
    # ========================================================
    # 1. Initialize the field in your dictionary
    product['mrp'] = ""
    
    # 2. Target elements that specifically represent the basis/strike-through price
    mrp_span = soup.select_one('span[class*="basisPrice"] span.a-offscreen')
    
    if not mrp_span:
        mrp_span = soup.select_one('span.a-price.a-text-price span.a-offscreen')
        
    if not mrp_span:
        # Fallback: Look for any small offscreen span, but verify it belongs to a price block
        # by ensuring its parent contains M.R.P. or List Price keywords
        potential_spans = soup.select('span.a-size-small.aok-offscreen')
        for span in potential_spans:
            parent_text = span.find_parent().get_text().lower() if span.find_parent() else ""
            if "m.r.p" in parent_text or "list price" in parent_text:
                mrp_span = span
                break

    if mrp_span:
        raw_mrp = mrp_span.get_text(strip=True)  # e.g., "M.R.P.: ₹15,000.00"
        try:
            # 1. Use Regex to find only digits, commas, and periods
            cleaned_numbers = re.sub(r'[^\d.,]', '', raw_mrp)
            
            # 2. Remove commas from the remaining number string
            cleaned_numbers = cleaned_numbers.replace(',', '')
            
            # 3. Convert to float first then to a clean integer
            product['mrp'] = int(float(cleaned_numbers))
        except:
            product['mrp'] = raw_mrp
    else:
        product['mrp'] = "Not found"

    # ========================================================
    # SEPARATE SECTION: DISCOUNT PERCENTAGE EXTRACTION
    # ========================================================
    # 1. Initialize the field in your dictionary
    product['discount_percent'] = " "
    
    # 2. Use a partial class match to find the savings percentage span
    # This matches any class that contains 'savingsPercentage'
    discount_span = soup.select_one('span[class*="savingsPercentage"]')
    
    if discount_span:
        raw_discount = discount_span.get_text(strip=True)  # e.g., "-70%"
        try:
            # Clean up text: remove minus signs, percent symbols, and whitespace
            # This removes standard hyphens, en-dashes, and em-dashes all at once
            clean_discount = raw_discount.replace('-', '').replace('–', '').replace('—', '').strip()
            product['discount_percent'] = int(clean_discount)  # Saves as an integer (70)
        except:
            # Fallback to the raw text (e.g., "-70%") if parsing fails
            product['discount_percent'] = raw_discount
    else:
        # If there is no discount span, it means the item is selling at full price
        product['discount_percent'] = 0

    product['review_count'] = 0  
    
    # 2. First check if the page explicitly states there are 0 reviews
    reviews_container = soup.select_one('#cr-top-reviews')
    is_zero = False
    
    if reviews_container:
        container_text = reviews_container.get_text(strip=True).lower()
        if "0 customer reviews" in container_text or "there are 0" in container_text:
            product['review_count'] = 0
            is_zero = True
    
    # 3. If it's not 0, grab the total count using the data-hook attribute
    if not is_zero:
        review_count_span = soup.select_one('span[data-hook="total-review-count"]')
        
        if review_count_span:
            raw_count = review_count_span.get_text(strip=True)  # e.g., "249 global ratings"
            try:
                # Removes commas for large numbers (e.g., "1,249" -> "1249") and extracts the digits
                clean_count = raw_count.split()[0].replace(',', '')
                product['review_count'] = int(clean_count)
            except:
                # Fallback to raw text if the splitting fails
                product['review_count'] = raw_count
        else:
            product['review_count'] = "Not found"
    
    # 1. Target the specific reviews container shown in your screenshot
    reviews_container = soup.select_one('#cr-top-reviews')
    
    is_zero_reviews = False
    
    if reviews_container:
        container_text = reviews_container.get_text(strip=True).lower()
        # Check specifically for "0 customer reviews" inside the review block
        if "0 customer reviews" in container_text or "there are 0" in container_text:
            product['rating'] = 0.0
            is_zero_reviews = True
    
    # 2. If it is NOT a zero-review product, safely proceed with your normal logic
    if not is_zero_reviews:
        rating_span = soup.select_one('#acrPopover span.a-icon-alt')
        
        if not rating_span:
            rating_span = soup.select_one('.a-icon.a-icon-star span')  
        
        if not rating_span:
            rating_span = soup.select_one('span.a-size-small.a-color-base')
    
        if rating_span:
            raw_rating = rating_span.get_text(strip=True)
            try:
                product['rating'] = float(raw_rating.split()[0])
            except:
                product['rating'] = raw_rating
        else:
            product['rating'] = ""
            print(f"[INFO] No rating found for: {product['url']}")


    images = get_amazon_images(url)  
    image_columns = images[:8] + [' '] * (8 - len(images))  
    
    for i in range(8):
        product[f'url{i+1}'] = image_columns[i]

    d = datetime.now()
    product['Scraped_Date'] = d.strftime("%d/%m.")
    # keywords = remove_non_english_words(get_keywords(product["description"]))
    # product['keywords'] = keywords
    product["seo_url"] = re.sub(r'[^a-z0-9]+', '-', product["title"].lower()).strip('-')
    
    return product

def scrape_page(page_num, start_sr_no):
    print(f"\nScraping page {page_num}")
    response = make_request(BASE_URL + str(page_num))
    if not response:
        print(f"Failed to fetch page {page_num}")
        return []

    soup = BeautifulSoup(response.content, 'html.parser')
    product_links = get_product_links(soup)
    print(f"Found {len(product_links)} products on page {page_num}")

    return [
        product for i, link in enumerate(product_links, start=start_sr_no)
        if (product := get_product_details(link, i)) is not None
    ]

def save_to_csv(data, filename=OUTPUT_FILE):
    df = pd.DataFrame(data)
    if all(col in df.columns for col in ['url1', 'url2', 'url3', 'url4', 'url5', 'url6', 'url7', 'url8']):
        df[' '] = ' '
    if df.empty:
        print("No new data to save.")
        return

    url_columns = ['url1', 'url2', 'url3', 'url4', 'url5', 'url6', 'url7', 'url8']
    if all(col in df.columns for col in url_columns):
        df[' '] = ' ' 

    if os.path.exists(filename):
        existing = pd.read_csv(filename)

        if 'Sr.No' in df.columns:
            df = df.drop(columns=['Sr.No'])

        df = df[~df['uid'].isin(existing['uid'])]

        if df.empty:
            print("No new unique products to add.")
            return

        start = existing.shape[0] + 1
        df.insert(0, 'Sr.No', range(start, start + len(df)))
        df = pd.concat([existing, df], ignore_index=True)
    else:
        if 'Sr.No' in df.columns:
            df = df.drop(columns=['Sr.No'])

        df.insert(0, 'Sr.No', range(1, len(df) + 1))

    df.to_csv(filename, index=False)
    print(f"[INFO] Saved {len(df)} new products to {filename}")


def main():
    all_products = []
    start_page, end_page = Start_Page_Num, End_Page_Num
    current_sr_no = 1

    for page in range(start_page, end_page + 1):
        products = scrape_page(page, current_sr_no)
        all_products.extend(products)
        current_sr_no += len(products)
        time.sleep(random.uniform(8, 15))

    save_to_csv(all_products)

if __name__ == '__main__':
    main()


Scraping page 16


KeyboardInterrupt: 